In [29]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Objective

The goal of this notebook is to complete Milestone 2 of the Smart MCQ Solver Challenge.
 
This notebook transitions from classical NLP (TF-IDF) to modern deep learning
architectures using the Hugging Face ecosystem.
 
This notebook covers:
 
- Hugging Face `datasets` & `transformers` library basics
- BERT tokenizer internals (vocab size, special tokens)
- Tokenizing prompts into PyTorch tensors
- BERT/RoBERTa architecture & attention head dimensionality
- Extracting `last_hidden_state` and `[CLS]` embeddings
- Visualizing attention weights
- Context-aware embeddings with Sentence-Transformers (MiniLM)
- Comparing TF-IDF vs MiniLM ranking pipelines (MAP@3)
- Zero-shot classification (softmax vs sigmoid)
- Generative QA with a small language model (Flan-T5)
 
⚠️ Enable GPU (T4 x2) and Internet in Kaggle notebook settings before running.
 
Evaluation Metric:
- Mean Average Precision @3 (MAP@3)

In [30]:
# !pip install -q transformers datasets sentence-transformers
 
import numpy as np
import pandas as pd
import torch
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
 
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

In [33]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
 
OPTIONS = ["A", "B", "C", "D", "E"]

In [34]:
# (Hugging Face Datasets & Tokenizer Basics)
 
# Load with HF `datasets` (not pandas)
 
hf_dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    split="train"
)

In [35]:
# Create combined_text column with .map()

def add_combined_text(example):
    example["combined_text"] = str(example["prompt"]) + " " + str(example["A"])
    return example

hf_dataset = hf_dataset.map(add_combined_text)

In [36]:
# Initialize BERT tokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [37]:
# Tokenize all prompts → PyTorch tensors

encoded = tokenizer(
    train["prompt"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [38]:
length_51 = len(hf_dataset[51]["combined_text"])
print(length_51)

614


In [39]:
vocab_size = tokenizer.vocab_size
print(vocab_size)

30522


In [40]:
sep_id = tokenizer.sep_token_id
print(sep_id)

102


In [41]:
print(encoded["input_ids"].shape)

torch.Size([2000, 128])


## BERT Architecture & Attention

In [42]:
# Load base model
bert_model = AutoModel.from_pretrained("bert-base-uncased")
bert_model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [43]:
hidden_size = 768
num_heads = 12

head_dim = hidden_size // num_heads
print(head_dim)

64


In [44]:
row0 = train[train["id"] == 1].iloc[0]

inputs = tokenizer(row0["prompt"], return_tensors="pt")

with torch.no_grad():
    output = bert_model(**inputs)

print(output.last_hidden_state.shape)

torch.Size([1, 31, 768])


In [45]:
cls_vector = output.last_hidden_state[0, 0, :]

first_five = cls_vector[:5].tolist()

print(round(sum(first_five), 4))

-1.2001


In [46]:
attn_model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)
attn_model.eval()

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

fusion_idx = tokens.index("fusion")

with torch.no_grad():
    attn_output = attn_model(**inputs)

last_layer = attn_output.attentions[-1]
head_0 = last_layer[0, 0]

weight = head_0[0, fusion_idx].item()
print(round(weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.1025


## Context-Aware Embeddings (Sentence Transformers)

In [47]:
# Load MiniLM model

minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [48]:
row0 = train[train["id"] == 1].iloc[0]

emb_prompt = minilm.encode(row0["prompt"], convert_to_tensor=True)
emb_optionB = minilm.encode(row0["B"], convert_to_tensor=True)

score = cos_sim(emb_prompt, emb_optionB).item()
print(round(score, 4))

0.7658


### TF-IDF pipeline

In [49]:
combined_text = []

for _, row in train.iterrows():
    combined_text.append(
        " ".join([row["prompt"], row["A"], row["B"], row["C"], row["D"], row["E"]])
    )

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

def tfidf_rank(row):
    prompt = vectorizer.transform([row["prompt"]])
    scores = {}
    for option in OPTIONS:
        option_vector = vectorizer.transform([row[option]])
        scores[option] = cosine_similarity(prompt, option_vector)[0][0]
    return sorted(scores, key=scores.get, reverse=True)

def map_at_3(answer, prediction):
    for rank, pred in enumerate(prediction[:3], start=1):
        if pred == answer:
            return 1 / rank
    return 0

tfidf_predictions = []
for _, row in train.iterrows():
    tfidf_predictions.append(tfidf_rank(row))

tfidf_scores = [
    map_at_3(train.iloc[i]["answer"], tfidf_predictions[i])
    for i in range(len(train))
]

print("TF-IDF MAP@3:", np.mean(tfidf_scores))

TF-IDF MAP@3: 0.2961666666666667


### MiniLM pipeline

In [50]:
prompt_embeddings = minilm.encode(
    train["prompt"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_tensor=True
)

option_embeddings = {}
for option in OPTIONS:
    option_embeddings[option] = minilm.encode(
        train[option].tolist(),
        batch_size=64,
        show_progress_bar=False,
        convert_to_tensor=True
    )

def minilm_rank(i):
    scores = {}
    for option in OPTIONS:
        scores[option] = cos_sim(
            prompt_embeddings[i],
            option_embeddings[option][i]
        ).item()
    return sorted(scores, key=scores.get, reverse=True)

minilm_predictions = []
for i in range(len(train)):
    minilm_predictions.append(minilm_rank(i))

minilm_scores = [
    map_at_3(train.iloc[i]["answer"], minilm_predictions[i])
    for i in range(len(train))
]

print("MiniLM MAP@3:", np.mean(minilm_scores))

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

MiniLM MAP@3: 0.4230833333333333


### Count: TF-IDF missed, MiniLM hit

In [51]:
count = 0

for i in range(len(train)):
    answer = train.iloc[i]["answer"]
    tfidf_hit = answer in tfidf_predictions[i][:3]
    minilm_hit = answer in minilm_predictions[i][:3]
    if not tfidf_hit and minilm_hit:
        count += 1

print(count)

502


### Zero-Shot Classification

In [52]:
# Load zero-shot pipeline

zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [53]:
row1 = train.iloc[1]

labels = [row1["A"], row1["B"], row1["C"]]

result = zero_shot(row1["prompt"], candidate_labels=labels)

print(result)

top_score = round(result["scores"][0], 4)
print(top_score)

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [54]:
result_multi = zero_shot(
    row1["prompt"],
    candidate_labels=labels,
    multi_label=True
)

print(result_multi)

softmax_sum = sum(result["scores"])
sigmoid_sum = sum(result_multi["scores"])

diff = abs(softmax_sum - sigmoid_sum)
print(round(diff, 4))

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion 

### Generative QA with Small Language Model

In [55]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# First row
row0 = train.iloc[0]

# Prompt
flan_prompt = f"""
Question: {row0['prompt']}

A. {row0['A']}
B. {row0['B']}

Answer with only A or B.
"""

# Tokenize
inputs = tokenizer(flan_prompt, return_tensors="pt")

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

# Decode output
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Model Output:", answer)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model Output: B


In [56]:
# row0 = train.iloc[0]

# flan_prompt = (
#     f"Question: {row0['prompt']}. "
#     f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
#     f"Answer with just the letter A or B."
# )

# output = flan(flan_prompt, max_new_tokens=5)

# print(output[0]["generated_text"])

### Submission

In [57]:
# Generate predictions on test.csv using MiniLM pipeline

test_prompt_embeddings = minilm.encode(
    test["prompt"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_tensor=True
)

test_option_embeddings = {}
for option in OPTIONS:
    test_option_embeddings[option] = minilm.encode(
        test[option].tolist(),
        batch_size=64,
        show_progress_bar=False,
        convert_to_tensor=True
    )

predictions = []

for i, row in test.iterrows():
    scores = {}
    for option in OPTIONS:
        scores[option] = cos_sim(
            test_prompt_embeddings[i],
            test_option_embeddings[option][i]
        ).item()
    ranked = sorted(scores, key=scores.get, reverse=True)
    predictions.append({"ID": row["id"], "Prediction": " ".join(ranked[:3])})

submission = pd.DataFrame(predictions)
submission.to_csv("submission.csv", index=False)

submission.head(10)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

,ID,Prediction
0,1,B E A
1,2,B D C
2,3,A C D
3,4,A E C
4,5,B C D
5,6,E A D
6,7,E C A
7,8,D A C
8,9,A C E
9,10,C E D
